# REST API Coding — Interview Prep

## Table of Contents

1. [HTTP Methods and Status Codes](#http-methods-and-status-codes) *(Beginner)*
2. [Making Requests with the `requests` Library](#making-requests-with-the-`requests`-library) *(Beginner)*
3. [Parsing JSON Responses](#parsing-json-responses) *(Beginner)*
4. [Query Parameters and Headers](#query-parameters-and-headers) *(Beginner)*
5. [API Authentication](#api-authentication) *(Mid-Level)*
6. [Pagination Handling](#pagination-handling) *(Mid-Level)*
7. [Error Handling and Retry Logic](#error-handling-and-retry-logic) *(Mid-Level)*
8. [Nested JSON Data](#nested-json-data) *(Mid-Level)*
9. [Building API Test Scripts](#building-api-test-scripts) *(Mid-Level)*


## Strategy Tips

## Strategy Tips for the REST API Coding Section (25 minutes)

**Read the Problem Fully:** Spend the first 1-2 minutes reading the entire problem statement. Pay close attention to the API endpoint structure, pagination details, and what the function must return.

**Start with Brute Force:** Get a working solution first. For pagination problems, a simple loop that fetches all pages and aggregates results is correct and scores full marks.

**Test Edge Cases:** Before submitting, consider: empty response lists, last page with fewer items, status codes other than 200, and deeply nested JSON keys.

**Time Management (2-3 problems in 25 minutes):**
- Problem 1 (easy): ~7 minutes
- Problem 2 (medium): ~10 minutes
- Problem 3 (harder): ~8 minutes
- If stuck on pagination logic for more than 4 minutes, write a working single-page version first, then add the loop.

**HackerRank REST API Patterns:**
- Most problems give you a base URL and ask you to fetch paginated data, filter it, and return an aggregated result.
- Always check `response.status_code == 200` before processing.
- Use `response.json()` to parse the body — it returns a dict or list.
- Pagination usually uses `?page=N` or `?offset=N&limit=M` query params.

**Common Pitfalls:**
- Forgetting to handle the last page (which may have fewer items)
- Using `response.text` instead of `response.json()`
- Not passing query parameters as a dict to `params=`
- Hardcoding page numbers instead of looping until empty

## HTTP Methods and Status Codes (Beginner)

### HTTP Methods and Status Codes

REST APIs communicate over HTTP. Understanding the semantics of each HTTP method and what status codes mean is foundational.

**HTTP Methods:**
- `GET` — Retrieve a resource. Should be idempotent and safe (no side effects).
- `POST` — Create a new resource. Body contains the new data.
- `PUT` — Replace an existing resource entirely.
- `PATCH` — Partially update an existing resource.
- `DELETE` — Remove a resource.

**Status Code Categories:**
- `2xx` — Success: `200 OK`, `201 Created`, `204 No Content`
- `3xx` — Redirection: `301 Moved Permanently`, `304 Not Modified`
- `4xx` — Client Error: `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `429 Too Many Requests`
- `5xx` — Server Error: `500 Internal Server Error`, `503 Service Unavailable`

**Key distinction:** `401 Unauthorized` means not authenticated; `403 Forbidden` means authenticated but not permitted.

In [ ]:
# --- HTTP method semantics with mock data ---
# Simulating what each HTTP method does conceptually

# Mock 'database' of users
users_db = [
    {'id': 1, 'name': 'Alice', 'role': 'admin'},
    {'id': 2, 'name': 'Bob',   'role': 'viewer'},
]

def mock_get(user_id: int) -> dict:
    """GET /users/{id} — retrieve a user."""
    for u in users_db:
        if u['id'] == user_id:
            return {'status': 200, 'body': u}
    return {'status': 404, 'body': {'error': 'Not found'}}

def mock_post(name: str, role: str) -> dict:
    """POST /users — create a new user."""
    new_id = max(u['id'] for u in users_db) + 1
    new_user = {'id': new_id, 'name': name, 'role': role}
    users_db.append(new_user)
    return {'status': 201, 'body': new_user}

def mock_delete(user_id: int) -> dict:
    """DELETE /users/{id} — remove a user."""
    global users_db
    before = len(users_db)
    users_db = [u for u in users_db if u['id'] != user_id]
    if len(users_db) < before:
        return {'status': 204, 'body': None}
    return {'status': 404, 'body': {'error': 'Not found'}}

print(mock_get(1))       # {'status': 200, 'body': {'id': 1, ...}}
print(mock_post('Carol', 'editor'))  # {'status': 201, ...}
print(mock_delete(99))   # {'status': 404, ...}

In [ ]:
# --- Status code handling pattern ---
def handle_response(status_code: int, body: dict) -> str:
    """Interpret a response based on its status code."""
    if 200 <= status_code < 300:
        return f'Success: {body}'
    elif status_code == 401:
        return 'Error: Not authenticated — check your credentials'
    elif status_code == 403:
        return 'Error: Forbidden — you lack permission'
    elif status_code == 404:
        return 'Error: Resource not found'
    elif status_code == 429:
        return 'Error: Rate limited — slow down requests'
    elif status_code >= 500:
        return f'Server error ({status_code}) — retry later'
    return f'Unexpected status: {status_code}'

print(handle_response(200, {'id': 1}))   # Success: ...
print(handle_response(404, {}))           # Error: Resource not found
print(handle_response(503, {}))           # Server error (503) ...

### Practice: Categorize HTTP Status Codes

Write a function that takes an HTTP status code (integer) and returns its category as a string: 'success', 'redirect', 'client_error', 'server_error', or 'unknown'.

**Function signature:** `def categorize_status(code: int) -> str:`

**Examples:**

- Input: `200` → Output: `'success'`
- Input: `404` → Output: `'client_error'`
- Input: `503` → Output: `'server_error'`
- Input: `301` → Output: `'redirect'`

**Hints:**

- Use range checks: 200 <= code < 300 for success, etc.
- Return 'unknown' as the fallback for anything outside 2xx-5xx.

In [ ]:
# TODO: Implement your solution
def categorize_status(code: int) -> str:
    pass

In [ ]:
# Solution: Categorize HTTP Status Codes
def categorize_status(code: int) -> str:
    """Return the category of an HTTP status code."""
    if 200 <= code < 300:
        return 'success'
    elif 300 <= code < 400:
        return 'redirect'
    elif 400 <= code < 500:
        return 'client_error'
    elif 500 <= code < 600:
        return 'server_error'
    return 'unknown'

In [ ]:
# Tests for Categorize HTTP Status Codes
assert categorize_status(200) == 'success'
assert categorize_status(201) == 'success'
assert categorize_status(204) == 'success'
assert categorize_status(301) == 'redirect'
assert categorize_status(404) == 'client_error'
assert categorize_status(401) == 'client_error'
assert categorize_status(500) == 'server_error'
assert categorize_status(503) == 'server_error'
assert categorize_status(100) == 'unknown'
print('All tests passed!')

### Practice: Filter Successful Responses

Given a list of API response dictionaries, each with a 'url' (str) and 'status_code' (int), return a list of URLs that returned a successful response (2xx status codes).

**Function signature:** `def successful_urls(responses: list[dict]) -> list[str]:`

**Examples:**

- Input: `[{'url': '/api/users', 'status_code': 200}, {'url': '/api/items', 'status_code': 404}, {'url': '/api/data', 'status_code': 201}]` → Output: `['/api/users', '/api/data']`

**Hints:**

- Use a list comprehension with a condition on status_code.
- 2xx means 200 <= code < 300.

In [ ]:
# TODO: Implement your solution
def successful_urls(responses: list[dict]) -> list[str]:
    pass

In [ ]:
# Solution: Filter Successful Responses
def successful_urls(responses: list[dict]) -> list[str]:
    """Return URLs with 2xx status codes."""
    return [
        r['url'] for r in responses
        if 200 <= r['status_code'] < 300
    ]

In [ ]:
# Tests for Filter Successful Responses
responses = [
    {'url': '/api/users', 'status_code': 200},
    {'url': '/api/items', 'status_code': 404},
    {'url': '/api/data',  'status_code': 201},
    {'url': '/api/admin', 'status_code': 403},
]
assert successful_urls(responses) == ['/api/users', '/api/data']
assert successful_urls([]) == []
assert successful_urls([{'url': '/x', 'status_code': 500}]) == []
print('All tests passed!')

### Practice: Count Errors by Category

Given a list of status codes (integers), return a dictionary with counts for each error category: 'client_errors' (4xx) and 'server_errors' (5xx). Ignore 2xx and 3xx codes.

**Function signature:** `def count_errors(status_codes: list[int]) -> dict[str, int]:`

**Examples:**

- Input: `[200, 404, 500, 403, 503, 201]` → Output: `{'client_errors': 2, 'server_errors': 2}`

**Hints:**

- Initialize the result dict with zero counts.
- Use range checks inside a for loop.

In [ ]:
# TODO: Implement your solution
def count_errors(status_codes: list[int]) -> dict[str, int]:
    pass

In [ ]:
# Solution: Count Errors by Category
def count_errors(status_codes: list[int]) -> dict[str, int]:
    """Count 4xx and 5xx errors in a list of status codes."""
    result = {'client_errors': 0, 'server_errors': 0}
    for code in status_codes:
        if 400 <= code < 500:
            result['client_errors'] += 1
        elif 500 <= code < 600:
            result['server_errors'] += 1
    return result

In [ ]:
# Tests for Count Errors by Category
assert count_errors([200, 404, 500, 403, 503, 201]) == {'client_errors': 2, 'server_errors': 2}
assert count_errors([200, 201, 204]) == {'client_errors': 0, 'server_errors': 0}
assert count_errors([]) == {'client_errors': 0, 'server_errors': 0}
assert count_errors([400, 401, 403, 404]) == {'client_errors': 4, 'server_errors': 0}
print('All tests passed!')

### Practice: Identify Idempotent Methods

Write a function that takes an HTTP method string and returns True if the method is idempotent (calling it multiple times produces the same result), False otherwise. Idempotent methods: GET, HEAD, PUT, DELETE, OPTIONS. Non-idempotent: POST, PATCH.

**Function signature:** `def is_idempotent(method: str) -> bool:`

**Examples:**

- Input: `'GET'` → Output: `True`
- Input: `'POST'` → Output: `False`
- Input: `'DELETE'` → Output: `True`

**Hints:**

- Store idempotent methods in a set for O(1) lookup.
- Normalize with .upper() to handle lowercase input.

In [ ]:
# TODO: Implement your solution
def is_idempotent(method: str) -> bool:
    pass

In [ ]:
# Solution: Identify Idempotent Methods
def is_idempotent(method: str) -> bool:
    """Return True if the HTTP method is idempotent."""
    idempotent_methods = {'GET', 'HEAD', 'PUT', 'DELETE', 'OPTIONS'}
    return method.upper() in idempotent_methods

In [ ]:
# Tests for Identify Idempotent Methods
assert is_idempotent('GET') == True
assert is_idempotent('HEAD') == True
assert is_idempotent('PUT') == True
assert is_idempotent('DELETE') == True
assert is_idempotent('OPTIONS') == True
assert is_idempotent('POST') == False
assert is_idempotent('PATCH') == False
assert is_idempotent('get') == True  # case-insensitive
print('All tests passed!')

### Key Takeaways

- GET retrieves, POST creates, PUT replaces, PATCH updates, DELETE removes.
- 2xx = success, 3xx = redirect, 4xx = client error, 5xx = server error.
- 401 means unauthenticated; 403 means authenticated but forbidden.
- GET, PUT, DELETE are idempotent; POST and PATCH are not.
- Always check the status code before processing the response body.

## Making Requests with the `requests` Library (Beginner)

### Making Requests with the `requests` Library

The `requests` library is the standard way to make HTTP calls in Python. It wraps the lower-level `urllib` with a clean, human-friendly API.

**Installation:** `pip install requests`

**Core functions:**
```python
import requests
r = requests.get(url, params={}, headers={}, timeout=10)
r = requests.post(url, json={}, headers={}, timeout=10)
r = requests.put(url, json={}, timeout=10)
r = requests.delete(url, timeout=10)
```

**Response object attributes:**
- `r.status_code` — integer status code (200, 404, etc.)
- `r.json()` — parse body as JSON (returns dict or list)
- `r.text` — body as a string
- `r.headers` — response headers dict
- `r.ok` — True if status_code < 400

**Always set a timeout** to avoid hanging indefinitely.

**Public test APIs used in examples:**
- `https://jsonplaceholder.typicode.com` — fake REST API for testing
- `https://httpbin.org` — HTTP request/response testing service

In [ ]:
# --- requests.get with JSONPlaceholder (mock equivalent) ---
# In a real environment you would call:
#   import requests
#   r = requests.get('https://jsonplaceholder.typicode.com/posts/1', timeout=10)
#   post = r.json()

# Mock equivalent for offline use:
def mock_get_post(post_id: int) -> dict:
    """Simulate GET /posts/{id} from JSONPlaceholder."""
    mock_data = {
        1: {'userId': 1, 'id': 1, 'title': 'sunt aut facere', 'body': 'quia et suscipit...'},
        2: {'userId': 1, 'id': 2, 'title': 'qui est esse',    'body': 'est rerum tempore...'},
    }
    if post_id in mock_data:
        return {'status_code': 200, 'json': mock_data[post_id]}
    return {'status_code': 404, 'json': {}}

response = mock_get_post(1)
if response['status_code'] == 200:
    post = response['json']
    print(f"Title: {post['title']}")
    print(f"User ID: {post['userId']}")

In [ ]:
# --- POST request pattern (mock) ---
# Real call:
#   r = requests.post(
#       'https://jsonplaceholder.typicode.com/posts',
#       json={'title': 'foo', 'body': 'bar', 'userId': 1},
#       timeout=10
#   )

def mock_post_create(payload: dict) -> dict:
    """Simulate POST /posts — JSONPlaceholder returns id=101 for new posts."""
    if not payload.get('title') or not payload.get('userId'):
        return {'status_code': 400, 'json': {'error': 'Missing required fields'}}
    created = {**payload, 'id': 101}
    return {'status_code': 201, 'json': created}

resp = mock_post_create({'title': 'New Post', 'body': 'Content', 'userId': 1})
print(f"Status: {resp['status_code']}")  # 201
print(f"Created ID: {resp['json']['id']}")  # 101

# Always check status before using the body
if resp['status_code'] in (200, 201):
    print('Resource created successfully')

### Practice: Extract Post Titles

Given a list of post dictionaries (simulating a GET /posts response from JSONPlaceholder), return a list of all post titles. Each post dict has keys: 'id', 'userId', 'title', 'body'.

**Function signature:** `def extract_titles(posts: list[dict]) -> list[str]:`

**Examples:**

- Input: `[{'id': 1, 'userId': 1, 'title': 'First Post', 'body': '...'}, {'id': 2, 'userId': 1, 'title': 'Second Post', 'body': '...'}]` → Output: `['First Post', 'Second Post']`

**Hints:**

- Use a list comprehension: [post['title'] for post in posts].

In [ ]:
# TODO: Implement your solution
def extract_titles(posts: list[dict]) -> list[str]:
    pass

In [ ]:
# Solution: Extract Post Titles
def extract_titles(posts: list[dict]) -> list[str]:
    """Extract title from each post dict."""
    return [post['title'] for post in posts]

In [ ]:
# Tests for Extract Post Titles
posts = [
    {'id': 1, 'userId': 1, 'title': 'First Post',  'body': 'body1'},
    {'id': 2, 'userId': 1, 'title': 'Second Post', 'body': 'body2'},
    {'id': 3, 'userId': 2, 'title': 'Third Post',  'body': 'body3'},
]
assert extract_titles(posts) == ['First Post', 'Second Post', 'Third Post']
assert extract_titles([]) == []
print('All tests passed!')

### Practice: Filter Posts by User

Given a list of post dictionaries and a user_id, return only the posts belonging to that user. Each post has a 'userId' field.

**Function signature:** `def posts_by_user(posts: list[dict], user_id: int) -> list[dict]:`

**Examples:**

- Input: `[{'id': 1, 'userId': 1, 'title': 'A'}, {'id': 2, 'userId': 2, 'title': 'B'}, {'id': 3, 'userId': 1, 'title': 'C'}], 1` → Output: `[{'id': 1, 'userId': 1, 'title': 'A'}, {'id': 3, 'userId': 1, 'title': 'C'}]`

**Hints:**

- Use a list comprehension with a condition on p['userId'].

In [ ]:
# TODO: Implement your solution
def posts_by_user(posts: list[dict], user_id: int) -> list[dict]:
    pass

In [ ]:
# Solution: Filter Posts by User
def posts_by_user(posts: list[dict], user_id: int) -> list[dict]:
    """Filter posts to only those belonging to user_id."""
    return [p for p in posts if p['userId'] == user_id]

In [ ]:
# Tests for Filter Posts by User
posts = [
    {'id': 1, 'userId': 1, 'title': 'A'},
    {'id': 2, 'userId': 2, 'title': 'B'},
    {'id': 3, 'userId': 1, 'title': 'C'},
]
result = posts_by_user(posts, 1)
assert len(result) == 2
assert all(p['userId'] == 1 for p in result)
assert posts_by_user(posts, 99) == []
assert posts_by_user([], 1) == []
print('All tests passed!')

### Practice: Build Request Summary

Given a list of response dictionaries, each with 'method' (str), 'url' (str), and 'status_code' (int), return a summary dict with keys 'total', 'success' (2xx), and 'failed' (4xx or 5xx).

**Function signature:** `def request_summary(responses: list[dict]) -> dict[str, int]:`

**Examples:**

- Input: `[{'method': 'GET', 'url': '/a', 'status_code': 200}, {'method': 'POST', 'url': '/b', 'status_code': 201}, {'method': 'GET', 'url': '/c', 'status_code': 404}]` → Output: `{'total': 3, 'success': 2, 'failed': 1}`

**Hints:**

- Use sum() with a generator expression for counting.
- 2xx: 200 <= code < 300; failed: code >= 400.

In [ ]:
# TODO: Implement your solution
def request_summary(responses: list[dict]) -> dict[str, int]:
    pass

In [ ]:
# Solution: Build Request Summary
def request_summary(responses: list[dict]) -> dict[str, int]:
    """Summarize a list of HTTP responses."""
    total = len(responses)
    success = sum(1 for r in responses if 200 <= r['status_code'] < 300)
    failed = sum(1 for r in responses if r['status_code'] >= 400)
    return {'total': total, 'success': success, 'failed': failed}

In [ ]:
# Tests for Build Request Summary
responses = [
    {'method': 'GET',  'url': '/a', 'status_code': 200},
    {'method': 'POST', 'url': '/b', 'status_code': 201},
    {'method': 'GET',  'url': '/c', 'status_code': 404},
    {'method': 'PUT',  'url': '/d', 'status_code': 500},
]
assert request_summary(responses) == {'total': 4, 'success': 2, 'failed': 2}
assert request_summary([]) == {'total': 0, 'success': 0, 'failed': 0}
print('All tests passed!')

### Practice: Find Slowest Endpoint

Given a list of API call records, each with 'endpoint' (str) and 'response_time_ms' (float), return the endpoint with the highest average response time. If the list is empty, return None.

**Function signature:** `def slowest_endpoint(records: list[dict]) -> str | None:`

**Examples:**

- Input: `[{'endpoint': '/api/users', 'response_time_ms': 120.0}, {'endpoint': '/api/posts', 'response_time_ms': 340.0}, {'endpoint': '/api/users', 'response_time_ms': 80.0}]` → Output: `'/api/posts'`

**Hints:**

- Group response times by endpoint using a dict of lists.
- Compute averages, then use max() with a key function.

In [ ]:
# TODO: Implement your solution
def slowest_endpoint(records: list[dict]) -> str | None:
    pass

In [ ]:
# Solution: Find Slowest Endpoint
def slowest_endpoint(records: list[dict]) -> str | None:
    """Return the endpoint with the highest average response time."""
    if not records:
        return None
    totals: dict[str, list[float]] = {}
    for r in records:
        ep = r['endpoint']
        totals.setdefault(ep, []).append(r['response_time_ms'])
    averages = {ep: sum(times) / len(times) for ep, times in totals.items()}
    return max(averages, key=averages.get)

In [ ]:
# Tests for Find Slowest Endpoint
records = [
    {'endpoint': '/api/users', 'response_time_ms': 120.0},
    {'endpoint': '/api/posts', 'response_time_ms': 340.0},
    {'endpoint': '/api/users', 'response_time_ms': 80.0},
]
assert slowest_endpoint(records) == '/api/posts'
assert slowest_endpoint([]) is None
single = [{'endpoint': '/x', 'response_time_ms': 50.0}]
assert slowest_endpoint(single) == '/x'
print('All tests passed!')

### Key Takeaways

- Use requests.get/post/put/delete with a timeout to avoid hanging.
- r.json() parses the response body; r.status_code gives the HTTP status.
- r.ok is True when status_code < 400 — a quick success check.
- Pass query parameters as a dict to params=, not in the URL string.
- JSONPlaceholder and httpbin.org are free public APIs for testing.

## Parsing JSON Responses (Beginner)

### Parsing JSON Responses

JSON (JavaScript Object Notation) is the universal data format for REST APIs. Python's `json` module and the `requests` library make parsing straightforward.

**Two ways to parse JSON:**
```python
import json
# From a string:
data = json.loads('{"key": "value"}')
# From a requests response:
data = response.json()
```

**Accessing nested data:**
```python
# Chained key access
city = data['address']['city']
# Safe access with .get()
city = data.get('address', {}).get('city', 'unknown')
```

**Common patterns:**
- `response.json()` returns a `dict` for object responses, `list` for arrays
- Use `json.dumps(data, indent=2)` to pretty-print for debugging
- Handle `json.JSONDecodeError` when the body might not be valid JSON

**Error handling:**
```python
try:
    data = response.json()
except ValueError:  # json.JSONDecodeError is a subclass
    data = {}
```

In [ ]:
# --- Parsing nested JSON (mock data) ---
import json

# Simulated API response body (as you'd get from response.json())
user_response = {
    'id': 1,
    'name': 'Leanne Graham',
    'email': 'sincere@april.biz',
    'address': {
        'street': 'Kulas Light',
        'city': 'Gwenborough',
        'zipcode': '92998-3874',
        'geo': {'lat': '-37.3159', 'lng': '81.1496'}
    },
    'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net'}
}

# Direct access
print(user_response['name'])                    # 'Leanne Graham'
print(user_response['address']['city'])         # 'Gwenborough'
print(user_response['address']['geo']['lat'])   # '-37.3159'

# Safe access with .get()
phone = user_response.get('phone', 'N/A')       # 'N/A' (key missing)
print(phone)

# Pretty-print for debugging
print(json.dumps(user_response['address'], indent=2))

In [ ]:
# --- Extracting fields from a list of JSON objects ---
posts = [
    {'id': 1, 'userId': 1, 'title': 'Post One',   'body': 'Content A'},
    {'id': 2, 'userId': 2, 'title': 'Post Two',   'body': 'Content B'},
    {'id': 3, 'userId': 1, 'title': 'Post Three', 'body': 'Content C'},
]

# Extract all titles
titles = [p['title'] for p in posts]
print(titles)  # ['Post One', 'Post Two', 'Post Three']

# Filter by userId and extract titles
user1_titles = [p['title'] for p in posts if p['userId'] == 1]
print(user1_titles)  # ['Post One', 'Post Three']

# Build a lookup dict: id -> title
id_to_title = {p['id']: p['title'] for p in posts}
print(id_to_title[2])  # 'Post Two'

### Practice: Extract Nested Field

Given a list of user dictionaries (each with a nested 'address' dict containing 'city'), return a list of city names. If a user has no 'address' or no 'city', use 'unknown'.

**Function signature:** `def extract_cities(users: list[dict]) -> list[str]:`

**Examples:**

- Input: `[{'id': 1, 'name': 'Alice', 'address': {'city': 'Boston'}}, {'id': 2, 'name': 'Bob', 'address': {'city': 'Austin'}}, {'id': 3, 'name': 'Carol'}]` → Output: `['Boston', 'Austin', 'unknown']`

**Hints:**

- Use .get('address', {}) to safely get the nested dict.
- Chain another .get('city', 'unknown') on the result.

In [ ]:
# TODO: Implement your solution
def extract_cities(users: list[dict]) -> list[str]:
    pass

In [ ]:
# Solution: Extract Nested Field
def extract_cities(users: list[dict]) -> list[str]:
    """Extract city from nested address, defaulting to 'unknown'."""
    return [
        u.get('address', {}).get('city', 'unknown')
        for u in users
    ]

In [ ]:
# Tests for Extract Nested Field
users = [
    {'id': 1, 'name': 'Alice', 'address': {'city': 'Boston'}},
    {'id': 2, 'name': 'Bob',   'address': {'city': 'Austin'}},
    {'id': 3, 'name': 'Carol'},
    {'id': 4, 'name': 'Dave',  'address': {}},
]
assert extract_cities(users) == ['Boston', 'Austin', 'unknown', 'unknown']
assert extract_cities([]) == []
print('All tests passed!')

### Practice: Flatten API Response

Given a JSON response dict with a 'data' key containing a list of items, and a 'meta' key with 'total' and 'page', return a flat dict with keys: 'items' (the list), 'total' (int), and 'page' (int).

**Function signature:** `def flatten_response(response: dict) -> dict:`

**Examples:**

- Input: `{'data': [{'id': 1}, {'id': 2}], 'meta': {'total': 50, 'page': 1}}` → Output: `{'items': [{'id': 1}, {'id': 2}], 'total': 50, 'page': 1}`

**Hints:**

- Use .get() with defaults for missing keys.
- Access nested 'meta' with response.get('meta', {}).

In [ ]:
# TODO: Implement your solution
def flatten_response(response: dict) -> dict:
    pass

In [ ]:
# Solution: Flatten API Response
def flatten_response(response: dict) -> dict:
    """Flatten a paginated API response into a simple dict."""
    return {
        'items': response.get('data', []),
        'total': response.get('meta', {}).get('total', 0),
        'page':  response.get('meta', {}).get('page', 1),
    }

In [ ]:
# Tests for Flatten API Response
resp = {'data': [{'id': 1}, {'id': 2}], 'meta': {'total': 50, 'page': 1}}
result = flatten_response(resp)
assert result['items'] == [{'id': 1}, {'id': 2}]
assert result['total'] == 50
assert result['page'] == 1
assert flatten_response({}) == {'items': [], 'total': 0, 'page': 1}
print('All tests passed!')

### Practice: Parse JSON String Safely

Write a function that takes a JSON string and returns the parsed Python object. If the string is not valid JSON, return an empty dict {}.

**Function signature:** `def safe_parse_json(json_str: str) -> dict | list:`

**Examples:**

- Input: `'{"key": "value"}'` → Output: `{'key': 'value'}`
- Input: `'not valid json'` → Output: `{}`
- Input: `''` → Output: `{}`

**Hints:**

- Use json.loads() inside a try/except block.
- Catch ValueError (which includes json.JSONDecodeError) and TypeError.

In [ ]:
# TODO: Implement your solution
def safe_parse_json(json_str: str) -> dict | list:
    pass

In [ ]:
# Solution: Parse JSON String Safely
import json

def safe_parse_json(json_str: str) -> dict | list:
    """Parse a JSON string, returning {} on failure."""
    try:
        return json.loads(json_str)
    except (ValueError, TypeError):
        return {}

In [ ]:
# Tests for Parse JSON String Safely
import json
assert safe_parse_json('{"key": "value"}') == {'key': 'value'}
assert safe_parse_json('[1, 2, 3]') == [1, 2, 3]
assert safe_parse_json('not valid json') == {}
assert safe_parse_json('') == {}
assert safe_parse_json('null') == {}
print('All tests passed!')

### Practice: Aggregate Scores from API Response

Given a list of test result dicts (each with 'test_name' and 'score' as a float 0-100), return a dict with 'average' (rounded to 2 decimal places), 'max_score', and 'min_score'. Return {'average': 0, 'max_score': 0, 'min_score': 0} for empty input.

**Function signature:** `def aggregate_scores(results: list[dict]) -> dict:`

**Examples:**

- Input: `[{'test_name': 'A', 'score': 90.0}, {'test_name': 'B', 'score': 75.5}, {'test_name': 'C', 'score': 88.0}]` → Output: `{'average': 84.5, 'max_score': 90.0, 'min_score': 75.5}`

**Hints:**

- Extract scores into a list first, then use sum(), max(), min().
- Use round(value, 2) for the average.

In [ ]:
# TODO: Implement your solution
def aggregate_scores(results: list[dict]) -> dict:
    pass

In [ ]:
# Solution: Aggregate Scores from API Response
def aggregate_scores(results: list[dict]) -> dict:
    """Compute average, max, and min scores from test results."""
    if not results:
        return {'average': 0, 'max_score': 0, 'min_score': 0}
    scores = [r['score'] for r in results]
    return {
        'average':   round(sum(scores) / len(scores), 2),
        'max_score': max(scores),
        'min_score': min(scores),
    }

In [ ]:
# Tests for Aggregate Scores from API Response
results = [
    {'test_name': 'A', 'score': 90.0},
    {'test_name': 'B', 'score': 75.5},
    {'test_name': 'C', 'score': 88.0},
]
agg = aggregate_scores(results)
assert agg['average'] == 84.5
assert agg['max_score'] == 90.0
assert agg['min_score'] == 75.5
assert aggregate_scores([]) == {'average': 0, 'max_score': 0, 'min_score': 0}
print('All tests passed!')

### Key Takeaways

- response.json() is the easiest way to parse a requests response body.
- json.loads() parses a JSON string; json.dumps() serializes to a string.
- Use .get('key', default) for safe access to avoid KeyError.
- Chain .get() calls for nested access: data.get('a', {}).get('b', 'default').
- Always handle json.JSONDecodeError (a subclass of ValueError) for robustness.

## Query Parameters and Headers (Beginner)

### Query Parameters and Headers

**Query parameters** are key-value pairs appended to a URL after `?`. They filter, sort, or paginate API results.

```python
# Don't build URLs manually:
# url = 'https://api.example.com/posts?userId=1&_limit=5'  # fragile

# Do use the params dict:
params = {'userId': 1, '_limit': 5}
r = requests.get('https://jsonplaceholder.typicode.com/posts', params=params)
# requests builds: .../posts?userId=1&_limit=5
```

**Headers** carry metadata about the request: content type, auth tokens, API keys, and more.

```python
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': 'Bearer <token>',
    'X-API-Key': '<api_key>',
}
r = requests.get(url, headers=headers)
```

**Common headers:**
- `Content-Type` — format of the request body
- `Accept` — format the client wants in the response
- `Authorization` — auth credentials (Bearer token, Basic auth)
- `X-API-Key` — custom API key header (varies by service)

In [ ]:
# --- Query parameters (mock simulation) ---
# Real call:
#   r = requests.get(
#       'https://jsonplaceholder.typicode.com/posts',
#       params={'userId': 1, '_limit': 3},
#       timeout=10
#   )

# Mock data simulating the filtered response
ALL_POSTS = [
    {'id': 1, 'userId': 1, 'title': 'Post A'},
    {'id': 2, 'userId': 2, 'title': 'Post B'},
    {'id': 3, 'userId': 1, 'title': 'Post C'},
    {'id': 4, 'userId': 1, 'title': 'Post D'},
    {'id': 5, 'userId': 3, 'title': 'Post E'},
]

def mock_get_posts(params: dict) -> list[dict]:
    """Simulate GET /posts with userId and _limit query params."""
    results = ALL_POSTS
    if 'userId' in params:
        results = [p for p in results if p['userId'] == params['userId']]
    if '_limit' in params:
        results = results[:params['_limit']]
    return results

filtered = mock_get_posts({'userId': 1, '_limit': 2})
print(filtered)  # [{'id': 1, 'userId': 1, 'title': 'Post A'}, ...]

In [ ]:
# --- Headers pattern ---
def build_auth_headers(api_key: str, content_type: str = 'application/json') -> dict:
    """Build standard request headers with API key auth."""
    return {
        'Content-Type': content_type,
        'Accept': 'application/json',
        'X-API-Key': api_key,
    }

headers = build_auth_headers('my-secret-key-123')
print(headers)
# {'Content-Type': 'application/json', 'Accept': 'application/json', 'X-API-Key': '...'}

# Bearer token pattern
def bearer_headers(token: str) -> dict:
    return {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
    }

print(bearer_headers('eyJhbGciOiJIUzI1NiJ9...'))

### Practice: Build Query String

Write a function that takes a base URL and a dict of query parameters, and returns the full URL with query string. Parameters should be sorted alphabetically by key for deterministic output.

**Function signature:** `def build_url(base_url: str, params: dict) -> str:`

**Examples:**

- Input: `'https://api.example.com/posts', {'userId': 1, '_limit': 5}` → Output: `'https://api.example.com/posts?_limit=5&userId=1'`
- Input: `'https://api.example.com/posts', {}` → Output: `'https://api.example.com/posts'`

**Hints:**

- Sort params.items() for deterministic output.
- Use '&'.join() to combine key=value pairs.

In [ ]:
# TODO: Implement your solution
def build_url(base_url: str, params: dict) -> str:
    pass

In [ ]:
# Solution: Build Query String
def build_url(base_url: str, params: dict) -> str:
    """Build a URL with sorted query parameters."""
    if not params:
        return base_url
    query_string = '&'.join(
        f'{k}={v}' for k, v in sorted(params.items())
    )
    return f'{base_url}?{query_string}'

In [ ]:
# Tests for Build Query String
assert build_url('https://api.example.com/posts', {'userId': 1, '_limit': 5}) == \
    'https://api.example.com/posts?_limit=5&userId=1'
assert build_url('https://api.example.com/posts', {}) == \
    'https://api.example.com/posts'
assert build_url('https://x.com/a', {'z': 3, 'a': 1}) == \
    'https://x.com/a?a=1&z=3'
print('All tests passed!')

### Practice: Validate Required Headers

Write a function that checks whether a headers dict contains all required headers. Return a list of missing header names (case-insensitive comparison). Return an empty list if all required headers are present.

**Function signature:** `def missing_headers(headers: dict, required: list[str]) -> list[str]:`

**Examples:**

- Input: `{'Content-Type': 'application/json', 'Authorization': 'Bearer x'}, ['Content-Type', 'Authorization', 'X-API-Key']` → Output: `['X-API-Key']`

**Hints:**

- Normalize both sides to lowercase for case-insensitive comparison.
- Use a set comprehension for the existing headers.

In [ ]:
# TODO: Implement your solution
def missing_headers(headers: dict, required: list[str]) -> list[str]:
    pass

In [ ]:
# Solution: Validate Required Headers
def missing_headers(headers: dict, required: list[str]) -> list[str]:
    """Return required headers not present in headers dict (case-insensitive)."""
    lower_headers = {k.lower() for k in headers}
    return [r for r in required if r.lower() not in lower_headers]

In [ ]:
# Tests for Validate Required Headers
h = {'Content-Type': 'application/json', 'Authorization': 'Bearer x'}
assert missing_headers(h, ['Content-Type', 'Authorization', 'X-API-Key']) == ['X-API-Key']
assert missing_headers(h, ['Content-Type', 'Authorization']) == []
assert missing_headers({}, ['Authorization']) == ['Authorization']
# Case-insensitive
assert missing_headers({'content-type': 'application/json'}, ['Content-Type']) == []
print('All tests passed!')

### Key Takeaways

- Always pass query parameters as a dict to params= — never build URLs manually.
- Headers carry metadata: Content-Type, Accept, Authorization, X-API-Key.
- Use f'Bearer {token}' for Bearer token auth headers.
- requests URL-encodes params automatically, handling special characters.
- Check required headers before sending requests to catch config errors early.

## API Authentication (Mid-Level)

### API Authentication

Most production APIs require authentication. The three most common patterns are API keys, Bearer tokens (OAuth2/JWT), and Basic auth.

**API Key (header or query param):**
```python
# Header-based (preferred)
headers = {'X-API-Key': 'your-api-key'}
r = requests.get(url, headers=headers)

# Query param (less secure — key visible in logs)
r = requests.get(url, params={'api_key': 'your-api-key'})
```

**Bearer Token (OAuth2/JWT):**
```python
headers = {'Authorization': f'Bearer {access_token}'}
r = requests.get(url, headers=headers)
```

**Basic Auth:**
```python
from requests.auth import HTTPBasicAuth
r = requests.get(url, auth=HTTPBasicAuth('username', 'password'))
# Or shorthand:
r = requests.get(url, auth=('username', 'password'))
```

**Redfish API (nip-relevant):** Redfish is a REST-based management API for server hardware (BMC/iDRAC). It uses Basic auth or session tokens and follows standard HTTP conventions.

**Token refresh pattern:** When a token expires (401 response), re-authenticate and retry the original request.

In [ ]:
# --- API Key and Bearer token patterns (mock) ---
def make_authenticated_request(
    url: str,
    auth_type: str,
    credential: str,
    mock_responses: dict,
) -> dict:
    """
    Simulate an authenticated request.
    auth_type: 'api_key' | 'bearer' | 'basic'
    """
    if auth_type == 'api_key':
        headers = {'X-API-Key': credential}
    elif auth_type == 'bearer':
        headers = {'Authorization': f'Bearer {credential}'}
    else:
        headers = {}

    # Simulate: valid credentials return 200, invalid return 401
    if credential == 'invalid':
        return {'status_code': 401, 'body': {'error': 'Unauthorized'}}
    return {'status_code': 200, 'body': mock_responses.get(url, {})}

mock_data = {'/api/systems': [{'id': 'sys1', 'status': 'OK'}]}
resp = make_authenticated_request('/api/systems', 'bearer', 'valid-token', mock_data)
print(resp)  # {'status_code': 200, 'body': [{'id': 'sys1', ...}]}

resp_bad = make_authenticated_request('/api/systems', 'bearer', 'invalid', mock_data)
print(resp_bad)  # {'status_code': 401, ...}

In [ ]:
# --- Redfish API interaction pattern (nip-relevant) ---
# Redfish is a REST API for server hardware management (BMC/iDRAC).
# It uses Basic auth and returns JSON following the DMTF Redfish schema.

MOCK_REDFISH_SYSTEMS = {
    '@odata.context': '/redfish/v1/$metadata#ComputerSystemCollection',
    '@odata.id': '/redfish/v1/Systems',
    'Members': [
        {'@odata.id': '/redfish/v1/Systems/1'},
        {'@odata.id': '/redfish/v1/Systems/2'},
    ],
    'Members@odata.count': 2,
}

MOCK_REDFISH_SYSTEM_1 = {
    '@odata.id': '/redfish/v1/Systems/1',
    'Id': '1',
    'Name': 'Compute Node 1',
    'Status': {'State': 'Enabled', 'Health': 'OK'},
    'PowerState': 'On',
    'ProcessorSummary': {'Count': 2, 'Model': 'Intel Xeon'},
    'MemorySummary': {'TotalSystemMemoryGiB': 256},
}

def get_redfish_system_health(system_data: dict) -> str:
    """Extract health status from a Redfish System resource."""
    return system_data.get('Status', {}).get('Health', 'Unknown')

def list_redfish_system_ids(collection: dict) -> list[str]:
    """Extract system IDs from a Redfish Systems collection."""
    return [
        member['@odata.id'].split('/')[-1]
        for member in collection.get('Members', [])
    ]

print(get_redfish_system_health(MOCK_REDFISH_SYSTEM_1))  # 'OK'
print(list_redfish_system_ids(MOCK_REDFISH_SYSTEMS))     # ['1', '2']

### Practice: Build Auth Headers

Write a function that builds an Authorization header dict based on the auth type. Supported types: 'bearer' (returns {'Authorization': 'Bearer <token>'}), 'api_key' (returns {'X-API-Key': '<token>'}), 'basic' (returns {'Authorization': 'Basic <token>'}). Raise ValueError for unknown auth types.

**Function signature:** `def build_auth_header(auth_type: str, token: str) -> dict:`

**Examples:**

- Input: `'bearer', 'abc123'` → Output: `{'Authorization': 'Bearer abc123'}`
- Input: `'api_key', 'xyz'` → Output: `{'X-API-Key': 'xyz'}`

**Hints:**

- Use if/elif for each auth type.
- Raise ValueError with a descriptive message for unknown types.

In [ ]:
# TODO: Implement your solution
def build_auth_header(auth_type: str, token: str) -> dict:
    pass

In [ ]:
# Solution: Build Auth Headers
def build_auth_header(auth_type: str, token: str) -> dict:
    """Build an auth header dict for the given auth type."""
    if auth_type == 'bearer':
        return {'Authorization': f'Bearer {token}'}
    elif auth_type == 'api_key':
        return {'X-API-Key': token}
    elif auth_type == 'basic':
        return {'Authorization': f'Basic {token}'}
    raise ValueError(f'Unknown auth type: {auth_type!r}')

In [ ]:
# Tests for Build Auth Headers
assert build_auth_header('bearer', 'abc123') == {'Authorization': 'Bearer abc123'}
assert build_auth_header('api_key', 'xyz') == {'X-API-Key': 'xyz'}
assert build_auth_header('basic', 'dXNlcjpwYXNz') == {'Authorization': 'Basic dXNlcjpwYXNz'}
try:
    build_auth_header('oauth', 'token')
    assert False, 'Should have raised ValueError'
except ValueError:
    pass
print('All tests passed!')

### Practice: Handle Token Expiry

Write a function that simulates token refresh logic. Given a response dict with 'status_code', if the status is 401, call the provided refresh_token() function (no args, returns a new token string) and return the new token. Otherwise return None (no refresh needed).

**Function signature:** `def refresh_if_expired(response: dict, refresh_token) -> str | None:`

**Examples:**

- Input: `{'status_code': 401, 'body': {}}, lambda: 'new-token-xyz'` → Output: `'new-token-xyz'`
- Input: `{'status_code': 200, 'body': {'data': []}}, lambda: 'new-token'` → Output: `None`

**Hints:**

- Check response['status_code'] == 401.
- Call refresh_token() (it's a callable passed as a parameter).

In [ ]:
# TODO: Implement your solution
def refresh_if_expired(response: dict, refresh_token) -> str | None:
    pass

In [ ]:
# Solution: Handle Token Expiry
def refresh_if_expired(response: dict, refresh_token) -> str | None:
    """Call refresh_token() if response is 401, else return None."""
    if response.get('status_code') == 401:
        return refresh_token()
    return None

In [ ]:
# Tests for Handle Token Expiry
resp_401 = {'status_code': 401, 'body': {}}
resp_200 = {'status_code': 200, 'body': {'data': []}}
assert refresh_if_expired(resp_401, lambda: 'new-token-xyz') == 'new-token-xyz'
assert refresh_if_expired(resp_200, lambda: 'new-token') is None
assert refresh_if_expired(resp_401, lambda: 'refreshed') == 'refreshed'
print('All tests passed!')

### Practice: Extract Redfish System Health

Given a list of Redfish System resource dicts (each with a nested 'Status' dict containing 'Health' and 'State'), return a list of dicts with 'id' (from the 'Id' field) and 'health' (from Status.Health). Use 'Unknown' if Health is missing.

**Function signature:** `def extract_system_health(systems: list[dict]) -> list[dict]:`

**Examples:**

- Input: `[{'Id': '1', 'Status': {'Health': 'OK', 'State': 'Enabled'}}, {'Id': '2', 'Status': {'Health': 'Warning', 'State': 'Enabled'}}, {'Id': '3'}]` → Output: `[{'id': '1', 'health': 'OK'}, {'id': '2', 'health': 'Warning'}, {'id': '3', 'health': 'Unknown'}]`

**Hints:**

- Use .get('Status', {}).get('Health', 'Unknown') for safe nested access.
- Build a list comprehension returning dicts.

In [ ]:
# TODO: Implement your solution
def extract_system_health(systems: list[dict]) -> list[dict]:
    pass

In [ ]:
# Solution: Extract Redfish System Health
def extract_system_health(systems: list[dict]) -> list[dict]:
    """Extract id and health from Redfish System resources."""
    return [
        {
            'id': s.get('Id', 'unknown'),
            'health': s.get('Status', {}).get('Health', 'Unknown'),
        }
        for s in systems
    ]

In [ ]:
# Tests for Extract Redfish System Health
systems = [
    {'Id': '1', 'Status': {'Health': 'OK',      'State': 'Enabled'}},
    {'Id': '2', 'Status': {'Health': 'Warning', 'State': 'Enabled'}},
    {'Id': '3'},
]
result = extract_system_health(systems)
assert result[0] == {'id': '1', 'health': 'OK'}
assert result[1] == {'id': '2', 'health': 'Warning'}
assert result[2] == {'id': '3', 'health': 'Unknown'}
assert extract_system_health([]) == []
print('All tests passed!')

### Practice: Mask Sensitive Headers

Write a function that takes a headers dict and returns a copy with sensitive header values replaced by '***'. Sensitive headers (case-insensitive): 'Authorization', 'X-API-Key', 'X-Auth-Token'.

**Function signature:** `def mask_sensitive_headers(headers: dict) -> dict:`

**Examples:**

- Input: `{'Content-Type': 'application/json', 'Authorization': 'Bearer secret', 'X-API-Key': 'my-key'}` → Output: `{'Content-Type': 'application/json', 'Authorization': '***', 'X-API-Key': '***'}`

**Hints:**

- Use a dict comprehension with a conditional expression.
- Normalize keys to lowercase for case-insensitive matching.

In [ ]:
# TODO: Implement your solution
def mask_sensitive_headers(headers: dict) -> dict:
    pass

In [ ]:
# Solution: Mask Sensitive Headers
def mask_sensitive_headers(headers: dict) -> dict:
    """Return headers dict with sensitive values masked."""
    sensitive = {'authorization', 'x-api-key', 'x-auth-token'}
    return {
        k: '***' if k.lower() in sensitive else v
        for k, v in headers.items()
    }

In [ ]:
# Tests for Mask Sensitive Headers
h = {
    'Content-Type': 'application/json',
    'Authorization': 'Bearer secret',
    'X-API-Key': 'my-key',
    'Accept': 'application/json',
}
masked = mask_sensitive_headers(h)
assert masked['Content-Type'] == 'application/json'
assert masked['Authorization'] == '***'
assert masked['X-API-Key'] == '***'
assert masked['Accept'] == 'application/json'
# Original unchanged
assert h['Authorization'] == 'Bearer secret'
print('All tests passed!')

### Key Takeaways

- Bearer tokens go in the Authorization header: 'Bearer <token>'.
- API keys can be in headers (X-API-Key) or query params — prefer headers.
- Basic auth encodes 'username:password' in base64 in the Authorization header.
- Handle 401 responses by refreshing the token and retrying.
- Redfish uses Basic auth or session tokens for server hardware management APIs.

## Pagination Handling (Mid-Level)

### Pagination Handling

APIs rarely return all results in one response. Pagination limits response size and protects server resources. You must loop to collect all data.

**Three common pagination styles:**

1. **Page-based:** `?page=1&per_page=20` — increment page until empty
```python
page = 1
all_items = []
while True:
    r = requests.get(url, params={'page': page, 'per_page': 20})
    items = r.json()
    if not items:
        break
    all_items.extend(items)
    page += 1
```

2. **Offset-based:** `?offset=0&limit=20` — increment offset by limit
```python
offset, limit = 0, 20
all_items = []
while True:
    r = requests.get(url, params={'offset': offset, 'limit': limit})
    items = r.json()
    if not items:
        break
    all_items.extend(items)
    offset += limit
```

3. **Cursor-based:** Response includes a `next_cursor` or `next` URL
```python
cursor = None
all_items = []
while True:
    params = {'cursor': cursor} if cursor else {}
    data = requests.get(url, params=params).json()
    all_items.extend(data['items'])
    cursor = data.get('next_cursor')
    if not cursor:
        break
```

**HackerRank pattern:** Most HackerRank REST API problems use page-based pagination. The loop terminates when the response is an empty list or when `page > total_pages`.

In [ ]:
# --- Page-based pagination (mock) ---
# Simulates a paginated API with 7 total items, 3 per page
ALL_USERS = [
    {'id': i, 'name': f'User {i}', 'active': i % 2 == 0}
    for i in range(1, 8)
]

def mock_paginated_api(page: int, per_page: int = 3) -> list[dict]:
    """Simulate GET /users?page=N&per_page=3."""
    start = (page - 1) * per_page
    return ALL_USERS[start:start + per_page]

def fetch_all_users() -> list[dict]:
    """Fetch all users across all pages."""
    all_users = []
    page = 1
    while True:
        users = mock_paginated_api(page)
        if not users:
            break
        all_users.extend(users)
        page += 1
    return all_users

all_users = fetch_all_users()
print(f'Total users fetched: {len(all_users)}')  # 7
print([u['id'] for u in all_users])              # [1, 2, 3, 4, 5, 6, 7]

In [ ]:
# --- Offset-based pagination with filtering (mock) ---
POSTS = [
    {'id': i, 'userId': (i % 3) + 1, 'title': f'Post {i}', 'views': i * 10}
    for i in range(1, 16)
]

def mock_posts_api(offset: int, limit: int = 5) -> list[dict]:
    """Simulate GET /posts?offset=N&limit=5."""
    return POSTS[offset:offset + limit]

def fetch_all_posts_paginated(limit: int = 5) -> list[dict]:
    """Collect all posts using offset pagination."""
    all_posts = []
    offset = 0
    while True:
        batch = mock_posts_api(offset, limit)
        if not batch:
            break
        all_posts.extend(batch)
        offset += limit
    return all_posts

all_posts = fetch_all_posts_paginated()
print(f'Total posts: {len(all_posts)}')  # 15

# Aggregate: total views across all posts
total_views = sum(p['views'] for p in all_posts)
print(f'Total views: {total_views}')

### Practice: Fetch All Pages

Given a function `get_page(page: int) -> list[dict]` that returns a page of items (empty list on last page), write a function that collects all items across all pages and returns them as a single list.

**Function signature:** `def fetch_all(get_page) -> list[dict]:`

**Examples:**

- Input: `get_page that returns [1,2,3] for page 1, [4,5] for page 2, [] for page 3` → Output: `[1, 2, 3, 4, 5]`

**Hints:**

- Start with page=1 and loop until get_page returns an empty list.
- Use list.extend() to add each page's items to the accumulator.

In [ ]:
# TODO: Implement your solution
def fetch_all(get_page) -> list[dict]:
    pass

In [ ]:
# Solution: Fetch All Pages
def fetch_all(get_page) -> list[dict]:
    """Collect all items from a paginated API."""
    all_items = []
    page = 1
    while True:
        items = get_page(page)
        if not items:
            break
        all_items.extend(items)
        page += 1
    return all_items

In [ ]:
# Tests for Fetch All Pages
pages = {1: [1, 2, 3], 2: [4, 5], 3: []}
def get_page(p): return pages.get(p, [])
assert fetch_all(get_page) == [1, 2, 3, 4, 5]

# Empty first page
assert fetch_all(lambda p: []) == []

# Single page
single = {1: ['a', 'b'], 2: []}
assert fetch_all(lambda p: single.get(p, [])) == ['a', 'b']
print('All tests passed!')

### Practice: Filter Across Paginated Results

Given a list of pages (each page is a list of user dicts with 'id', 'name', 'active'), return a list of names of all active users across all pages.

**Function signature:** `def active_users_from_pages(pages: list[list[dict]]) -> list[str]:`

**Examples:**

- Input: `[[{'id': 1, 'name': 'Alice', 'active': True}, {'id': 2, 'name': 'Bob', 'active': False}], [{'id': 3, 'name': 'Carol', 'active': True}]]` → Output: `['Alice', 'Carol']`

**Hints:**

- Use a nested list comprehension: for page in pages, for user in page.
- Add an if condition to filter active users.

In [ ]:
# TODO: Implement your solution
def active_users_from_pages(pages: list[list[dict]]) -> list[str]:
    pass

In [ ]:
# Solution: Filter Across Paginated Results
def active_users_from_pages(pages: list[list[dict]]) -> list[str]:
    """Collect names of active users across all pages."""
    return [
        user['name']
        for page in pages
        for user in page
        if user.get('active')
    ]

In [ ]:
# Tests for Filter Across Paginated Results
pages = [
    [{'id': 1, 'name': 'Alice', 'active': True},
     {'id': 2, 'name': 'Bob',   'active': False}],
    [{'id': 3, 'name': 'Carol', 'active': True},
     {'id': 4, 'name': 'Dave',  'active': False}],
]
assert active_users_from_pages(pages) == ['Alice', 'Carol']
assert active_users_from_pages([]) == []
assert active_users_from_pages([[{'id': 1, 'name': 'X', 'active': False}]]) == []
print('All tests passed!')

### Practice: Aggregate Paginated Totals

Given a function `get_page(page: int) -> dict` that returns {'items': [...], 'total_pages': N}, write a function that fetches all pages and returns the total count of items.

**Function signature:** `def count_all_items(get_page) -> int:`

**Examples:**

- Input: `get_page returning 3 items on page 1, 2 items on page 2, total_pages=2` → Output: `5`

**Hints:**

- Check page >= total_pages to know when to stop.
- Accumulate len(items) each iteration.

In [ ]:
# TODO: Implement your solution
def count_all_items(get_page) -> int:
    pass

In [ ]:
# Solution: Aggregate Paginated Totals
def count_all_items(get_page) -> int:
    """Count total items across all pages."""
    total = 0
    page = 1
    while True:
        data = get_page(page)
        items = data.get('items', [])
        total += len(items)
        if page >= data.get('total_pages', 1):
            break
        page += 1
    return total

In [ ]:
# Tests for Aggregate Paginated Totals
def make_api(pages_data):
    total = len(pages_data)
    def get_page(p):
        return {'items': pages_data[p - 1], 'total_pages': total}
    return get_page

api = make_api([[1, 2, 3], [4, 5]])
assert count_all_items(api) == 5

api_single = make_api([[10, 20]])
assert count_all_items(api_single) == 2

api_empty = make_api([[]])
assert count_all_items(api_empty) == 0
print('All tests passed!')

### Practice: Find Item Across Pages

Given a list of pages (each a list of dicts with 'id' and 'value'), and a target id, return the 'value' of the item with that id. Return None if not found.

**Function signature:** `def find_across_pages(pages: list[list[dict]], target_id: int) -> object:`

**Examples:**

- Input: `[[{'id': 1, 'value': 'a'}, {'id': 2, 'value': 'b'}], [{'id': 3, 'value': 'c'}]], 2` → Output: `'b'`
- Input: `[[{'id': 1, 'value': 'a'}]], 99` → Output: `None`

**Hints:**

- Use nested for loops: for page in pages, for item in page.
- Return early when found; return None after all pages exhausted.

In [ ]:
# TODO: Implement your solution
def find_across_pages(pages: list[list[dict]], target_id: int) -> object:
    pass

In [ ]:
# Solution: Find Item Across Pages
def find_across_pages(pages: list[list[dict]], target_id: int) -> object:
    """Search for an item by id across all pages."""
    for page in pages:
        for item in page:
            if item['id'] == target_id:
                return item['value']
    return None

In [ ]:
# Tests for Find Item Across Pages
pages = [
    [{'id': 1, 'value': 'a'}, {'id': 2, 'value': 'b'}],
    [{'id': 3, 'value': 'c'}],
]
assert find_across_pages(pages, 2) == 'b'
assert find_across_pages(pages, 3) == 'c'
assert find_across_pages(pages, 99) is None
assert find_across_pages([], 1) is None
print('All tests passed!')

### Key Takeaways

- Always loop to collect all pages — never assume one response has everything.
- Page-based: increment page until empty list. Offset-based: increment offset by limit.
- Cursor-based pagination uses a token from the response to fetch the next page.
- Use list.extend() (not append) to add a page's items to the accumulator.
- HackerRank REST problems almost always involve pagination — master this pattern.

## Error Handling and Retry Logic (Mid-Level)

### Error Handling and Retry Logic

Robust API clients handle failures gracefully. Network issues, rate limits, and transient server errors are common in production.

**Status code checking:**
```python
r = requests.get(url, timeout=10)
r.raise_for_status()  # raises HTTPError for 4xx/5xx
# Or manually:
if r.status_code != 200:
    raise ValueError(f'API error: {r.status_code}')
```

**Timeout handling:**
```python
try:
    r = requests.get(url, timeout=5)
except requests.Timeout:
    print('Request timed out')
except requests.ConnectionError:
    print('Network error')
```

**Exponential backoff retry:**
```python
import time

def get_with_retry(url, max_retries=3, backoff=1.0):
    for attempt in range(max_retries):
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:  # rate limited
            time.sleep(backoff * (2 ** attempt))
        else:
            break  # non-retryable error
    raise RuntimeError(f'Failed after {max_retries} attempts')
```

**Retryable vs non-retryable errors:**
- Retry: `429 Too Many Requests`, `503 Service Unavailable`, network timeouts
- Don't retry: `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`

In [ ]:
# --- Retry with exponential backoff (mock) ---
import time

def get_with_retry(
    fetch_fn,          # callable(attempt) -> dict with 'status_code'
    max_retries: int = 3,
    base_delay: float = 0.0,  # 0 for tests; use 1.0 in production
) -> dict:
    """
    Retry a request with exponential backoff on 429/5xx errors.
    Returns the response dict on success.
    Raises RuntimeError after max_retries exhausted.
    """
    retryable = {429, 500, 502, 503, 504}
    for attempt in range(max_retries):
        response = fetch_fn(attempt)
        code = response['status_code']
        if code == 200:
            return response
        if code in retryable:
            delay = base_delay * (2 ** attempt)
            if delay > 0:
                time.sleep(delay)
            continue
        # Non-retryable (4xx except 429)
        raise ValueError(f'Non-retryable error: {code}')
    raise RuntimeError(f'Failed after {max_retries} attempts')

# Simulate: fails twice with 503, then succeeds
call_count = [0]
def flaky_api(attempt):
    call_count[0] += 1
    if call_count[0] < 3:
        return {'status_code': 503, 'body': {}}
    return {'status_code': 200, 'body': {'data': 'success'}}

result = get_with_retry(flaky_api, max_retries=3, base_delay=0)
print(result)          # {'status_code': 200, 'body': {'data': 'success'}}
print(call_count[0])   # 3

In [ ]:
# --- Comprehensive error handling pattern ---
def safe_api_call(fetch_fn, url: str) -> dict | None:
    """
    Make an API call with full error handling.
    Returns parsed JSON on success, None on failure.
    """
    try:
        response = fetch_fn(url)
        status = response.get('status_code', 0)

        if status == 200:
            return response.get('body')
        elif status == 401:
            print(f'Authentication failed for {url}')
        elif status == 403:
            print(f'Access forbidden: {url}')
        elif status == 404:
            print(f'Resource not found: {url}')
        elif status == 429:
            print(f'Rate limited — slow down requests to {url}')
        elif status >= 500:
            print(f'Server error {status} for {url}')
        return None
    except Exception as e:
        print(f'Request failed: {e}')
        return None

# Test it
def mock_fetch(url):
    return {'status_code': 200, 'body': {'result': 42}}

data = safe_api_call(mock_fetch, '/api/data')
print(data)  # {'result': 42}

### Practice: Retry Until Success

Write a function that calls `fetch()` (no args, returns a dict with 'status_code') up to `max_retries` times. Return the response dict when status_code is 200. If all attempts fail, return the last response dict.

**Function signature:** `def retry_until_success(fetch, max_retries: int = 3) -> dict:`

**Examples:**

- Input: `fetch that fails twice then returns 200, max_retries=3` → Output: `{'status_code': 200, 'body': 'ok'}`

**Hints:**

- Loop max_retries times, return early on status 200.
- Keep track of the last response to return if all attempts fail.

In [ ]:
# TODO: Implement your solution
def retry_until_success(fetch, max_retries: int = 3) -> dict:
    pass

In [ ]:
# Solution: Retry Until Success
def retry_until_success(fetch, max_retries: int = 3) -> dict:
    """Retry fetch() up to max_retries times, return first 200 or last response."""
    response = {'status_code': 0}
    for _ in range(max_retries):
        response = fetch()
        if response.get('status_code') == 200:
            return response
    return response

In [ ]:
# Tests for Retry Until Success
# Succeeds on 3rd attempt
attempts = [0]
def flaky():
    attempts[0] += 1
    if attempts[0] < 3:
        return {'status_code': 503}
    return {'status_code': 200, 'body': 'ok'}

result = retry_until_success(flaky, max_retries=3)
assert result['status_code'] == 200

# Always fails — returns last response
always_fail = lambda: {'status_code': 500}
result2 = retry_until_success(always_fail, max_retries=2)
assert result2['status_code'] == 500
print('All tests passed!')

### Practice: Classify Retryable Errors

Write a function that takes a status code and returns True if the error is retryable, False otherwise. Retryable codes: 429, 500, 502, 503, 504. All other codes are not retryable.

**Function signature:** `def is_retryable(status_code: int) -> bool:`

**Examples:**

- Input: `429` → Output: `True`
- Input: `503` → Output: `True`
- Input: `404` → Output: `False`
- Input: `200` → Output: `False`

**Hints:**

- Use a set for O(1) membership testing.

In [ ]:
# TODO: Implement your solution
def is_retryable(status_code: int) -> bool:
    pass

In [ ]:
# Solution: Classify Retryable Errors
def is_retryable(status_code: int) -> bool:
    """Return True if the status code indicates a retryable error."""
    return status_code in {429, 500, 502, 503, 504}

In [ ]:
# Tests for Classify Retryable Errors
assert is_retryable(429) == True
assert is_retryable(500) == True
assert is_retryable(502) == True
assert is_retryable(503) == True
assert is_retryable(504) == True
assert is_retryable(200) == False
assert is_retryable(404) == False
assert is_retryable(401) == False
assert is_retryable(403) == False
print('All tests passed!')

### Practice: Compute Backoff Delay

Write a function that computes the exponential backoff delay in seconds for a given attempt number (0-indexed). Formula: `base * (2 ** attempt)`. Cap the result at `max_delay` seconds.

**Function signature:** `def backoff_delay(attempt: int, base: float = 1.0, max_delay: float = 60.0) -> float:`

**Examples:**

- Input: `0, base=1.0` → Output: `1.0`
- Input: `3, base=1.0` → Output: `8.0`
- Input: `10, base=1.0, max_delay=60.0` → Output: `60.0`

**Hints:**

- Use min() to cap the result at max_delay.
- Formula: base * (2 ** attempt).

In [ ]:
# TODO: Implement your solution
def backoff_delay(attempt: int, base: float = 1.0, max_delay: float = 60.0) -> float:
    pass

In [ ]:
# Solution: Compute Backoff Delay
def backoff_delay(attempt: int, base: float = 1.0, max_delay: float = 60.0) -> float:
    """Compute exponential backoff delay, capped at max_delay."""
    return min(base * (2 ** attempt), max_delay)

In [ ]:
# Tests for Compute Backoff Delay
assert backoff_delay(0) == 1.0
assert backoff_delay(1) == 2.0
assert backoff_delay(2) == 4.0
assert backoff_delay(3) == 8.0
assert backoff_delay(10) == 60.0  # capped
assert backoff_delay(0, base=0.5) == 0.5
assert backoff_delay(3, base=2.0, max_delay=10.0) == 10.0  # capped
print('All tests passed!')

### Practice: Parse Error Response

Given an API error response dict with 'status_code' and optionally 'error' (str) and 'message' (str), return a human-readable error string in the format: 'HTTP {code}: {error} - {message}'. Use 'Unknown Error' if 'error' is missing, and 'No details' if 'message' is missing.

**Function signature:** `def format_error(response: dict) -> str:`

**Examples:**

- Input: `{'status_code': 404, 'error': 'Not Found', 'message': 'User does not exist'}` → Output: `'HTTP 404: Not Found - User does not exist'`
- Input: `{'status_code': 500}` → Output: `'HTTP 500: Unknown Error - No details'`

**Hints:**

- Use .get() with default values for optional fields.
- Use an f-string to format the output.

In [ ]:
# TODO: Implement your solution
def format_error(response: dict) -> str:
    pass

In [ ]:
# Solution: Parse Error Response
def format_error(response: dict) -> str:
    """Format an API error response as a human-readable string."""
    code = response.get('status_code', 0)
    error = response.get('error', 'Unknown Error')
    message = response.get('message', 'No details')
    return f'HTTP {code}: {error} - {message}'

In [ ]:
# Tests for Parse Error Response
r1 = {'status_code': 404, 'error': 'Not Found', 'message': 'User does not exist'}
assert format_error(r1) == 'HTTP 404: Not Found - User does not exist'

r2 = {'status_code': 500}
assert format_error(r2) == 'HTTP 500: Unknown Error - No details'

r3 = {'status_code': 401, 'error': 'Unauthorized'}
assert format_error(r3) == 'HTTP 401: Unauthorized - No details'
print('All tests passed!')

### Key Takeaways

- Always set a timeout on requests to avoid hanging indefinitely.
- Retryable errors: 429, 500, 502, 503, 504. Non-retryable: 400, 401, 403, 404.
- Exponential backoff: delay = base * 2^attempt — prevents overwhelming the server.
- Use raise_for_status() for concise error checking, or check status_code manually.
- Catch requests.Timeout and requests.ConnectionError for network-level failures.

## Nested JSON Data (Mid-Level)

### Nested JSON Data

Real-world API responses are often deeply nested. Mastering techniques for safe access, flattening, and field extraction is essential.

**Deep access patterns:**
```python
# Chained access (raises KeyError if any key missing)
city = data['user']['address']['city']

# Safe chained access
city = data.get('user', {}).get('address', {}).get('city', 'N/A')

# Using a helper for arbitrary depth
def deep_get(d, *keys, default=None):
    for key in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(key, default)
    return d

city = deep_get(data, 'user', 'address', 'city', default='N/A')
```

**Flattening nested structures:**
```python
# Flatten a list of lists
flat = [item for sublist in nested for item in sublist]

# Flatten a dict of lists
all_items = [item for items in d.values() for item in items]
```

**Extracting fields from nested arrays:**
```python
# Get all tag names from posts with nested tags list
all_tags = [tag for post in posts for tag in post.get('tags', [])]
```

In [ ]:
# --- Deep access helper ---
def deep_get(d: dict, *keys, default=None):
    """Safely access nested dict keys."""
    for key in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(key, default)
        if d is default:
            return default
    return d

# Complex nested response (e.g., from a CI/CD API)
build_response = {
    'build': {
        'id': 'build-42',
        'status': 'failed',
        'stages': [
            {'name': 'compile', 'result': 'passed', 'duration_s': 12},
            {'name': 'test',    'result': 'failed', 'duration_s': 45},
            {'name': 'deploy',  'result': 'skipped','duration_s': 0},
        ],
        'metadata': {
            'triggered_by': 'push',
            'branch': 'main',
            'commit': {'sha': 'abc123', 'author': 'alice'},
        },
    }
}

print(deep_get(build_response, 'build', 'status'))              # 'failed'
print(deep_get(build_response, 'build', 'metadata', 'branch'))  # 'main'
print(deep_get(build_response, 'build', 'metadata', 'commit', 'author'))  # 'alice'
print(deep_get(build_response, 'build', 'missing_key', default='N/A'))    # 'N/A'

# Extract failed stages
stages = build_response['build']['stages']
failed = [s['name'] for s in stages if s['result'] == 'failed']
print(failed)  # ['test']

In [ ]:
# --- Flattening nested arrays ---
# API response: users with multiple roles
users_with_roles = [
    {'id': 1, 'name': 'Alice', 'roles': ['admin', 'developer']},
    {'id': 2, 'name': 'Bob',   'roles': ['viewer']},
    {'id': 3, 'name': 'Carol', 'roles': ['developer', 'tester']},
]

# All unique roles across all users
all_roles = list({role for user in users_with_roles for role in user['roles']})
print(sorted(all_roles))  # ['admin', 'developer', 'tester', 'viewer']

# Users who have the 'developer' role
devs = [u['name'] for u in users_with_roles if 'developer' in u['roles']]
print(devs)  # ['Alice', 'Carol']

# Flatten: list of (user_name, role) pairs
pairs = [(u['name'], role) for u in users_with_roles for role in u['roles']]
print(pairs[:3])  # [('Alice', 'admin'), ('Alice', 'developer'), ('Bob', 'viewer')]

### Practice: Deep Get Utility

Implement a `deep_get(d, *keys, default=None)` function that safely traverses a nested dict using the provided keys. Return `default` if any key is missing or if an intermediate value is not a dict.

**Function signature:** `def deep_get(d: dict, *keys, default=None):`

**Examples:**

- Input: `{'a': {'b': {'c': 42}}}, 'a', 'b', 'c'` → Output: `42`
- Input: `{'a': {'b': 1}}, 'a', 'x', default='missing'` → Output: `'missing'`

**Hints:**

- Iterate through keys, updating current at each step.
- Check isinstance(current, dict) before calling .get().

In [ ]:
# TODO: Implement your solution
def deep_get(d: dict, *keys, default=None):
    pass

In [ ]:
# Solution: Deep Get Utility
def deep_get(d: dict, *keys, default=None):
    """Safely traverse nested dicts, returning default if any key is missing."""
    current = d
    for key in keys:
        if not isinstance(current, dict):
            return default
        current = current.get(key, default)
        if current is default:
            return default
    return current

In [ ]:
# Tests for Deep Get Utility
assert deep_get({'a': {'b': {'c': 42}}}, 'a', 'b', 'c') == 42
assert deep_get({'a': {'b': 1}}, 'a', 'x', default='missing') == 'missing'
assert deep_get({}, 'a', default=0) == 0
assert deep_get({'a': None}, 'a', 'b', default='N/A') == 'N/A'
assert deep_get({'x': 5}, 'x') == 5
print('All tests passed!')

### Practice: Extract All Tags

Given a list of article dicts, each with a 'tags' key containing a list of strings, return a sorted list of all unique tags across all articles.

**Function signature:** `def all_unique_tags(articles: list[dict]) -> list[str]:`

**Examples:**

- Input: `[{'title': 'A', 'tags': ['python', 'api']}, {'title': 'B', 'tags': ['api', 'rest']}, {'title': 'C', 'tags': ['python']}]` → Output: `['api', 'python', 'rest']`

**Hints:**

- Use a set comprehension with nested iteration.
- Use .get('tags', []) to handle missing tags key.

In [ ]:
# TODO: Implement your solution
def all_unique_tags(articles: list[dict]) -> list[str]:
    pass

In [ ]:
# Solution: Extract All Tags
def all_unique_tags(articles: list[dict]) -> list[str]:
    """Return sorted unique tags from all articles."""
    return sorted({tag for a in articles for tag in a.get('tags', [])})

In [ ]:
# Tests for Extract All Tags
articles = [
    {'title': 'A', 'tags': ['python', 'api']},
    {'title': 'B', 'tags': ['api', 'rest']},
    {'title': 'C', 'tags': ['python']},
]
assert all_unique_tags(articles) == ['api', 'python', 'rest']
assert all_unique_tags([]) == []
assert all_unique_tags([{'title': 'X'}]) == []  # no tags key
print('All tests passed!')

### Key Takeaways

- Use .get('key', {}) chaining for safe nested access without KeyError.
- A deep_get() helper simplifies access to arbitrary-depth nested keys.
- Flatten nested lists with: [item for sublist in nested for item in sublist].
- Use set comprehensions to collect unique values from nested arrays.
- Always use .get('key', []) when accessing list fields that may be absent.

## Building API Test Scripts (Mid-Level)

### Building API Test Scripts

API test scripts validate that endpoints behave correctly. They combine HTTP calls with assertions to catch regressions.

**Core test patterns:**
```python
def test_get_user():
    response = requests.get(f'{BASE_URL}/users/1', timeout=10)
    assert response.status_code == 200
    data = response.json()
    assert 'id' in data
    assert data['id'] == 1
```

**Mock responses for offline testing:**
```python
from unittest.mock import patch, Mock

def test_with_mock():
    mock_resp = Mock()
    mock_resp.status_code = 200
    mock_resp.json.return_value = {'id': 1, 'name': 'Alice'}
    with patch('requests.get', return_value=mock_resp):
        result = my_api_function()
    assert result['name'] == 'Alice'
```

**nip-relevant patterns:**
- **Redfish API test:** Validate server health endpoints
- **CI/CD webhook handler:** Process build event payloads
- **Test result reporting client:** POST results to a reporting API

**HackerRank REST API patterns:**
- Fetch all pages, filter by a condition, return count or list
- Aggregate a numeric field (sum, average, max) across all pages
- Join data from two endpoints (e.g., users + posts)

In [ ]:
# --- CI/CD webhook handler pattern (nip-relevant) ---
# A webhook handler receives POST requests from CI/CD systems
# (e.g., Jenkins, GitHub Actions) when build events occur.

def handle_webhook_payload(payload: dict) -> dict:
    """
    Process a CI/CD webhook payload.
    Returns a summary dict with build status and key metadata.
    """
    event_type = payload.get('event', 'unknown')
    build = payload.get('build', {})

    summary = {
        'event': event_type,
        'build_id': build.get('id'),
        'status': build.get('status', 'unknown'),
        'branch': build.get('ref', 'unknown'),
        'duration_s': build.get('duration', 0),
        'failed_stages': [
            s['name'] for s in build.get('stages', [])
            if s.get('result') == 'failed'
        ],
    }
    return summary

# Simulate a Jenkins/GitHub Actions webhook payload
webhook_payload = {
    'event': 'build_complete',
    'build': {
        'id': 'build-99',
        'status': 'failed',
        'ref': 'feature/new-driver',
        'duration': 127,
        'stages': [
            {'name': 'lint',    'result': 'passed'},
            {'name': 'test',    'result': 'failed'},
            {'name': 'package', 'result': 'skipped'},
        ],
    }
}

result = handle_webhook_payload(webhook_payload)
print(result['status'])         # 'failed'
print(result['failed_stages'])  # ['test']

In [ ]:
# --- Test result reporting API client (nip-relevant) ---
# Posts test results to a reporting API (e.g., TestRail, custom dashboard)

def format_test_results_payload(suite_name: str, results: list[dict]) -> dict:
    """
    Build a payload for posting test results to a reporting API.
    Each result dict has: 'name', 'status' ('pass'/'fail'), 'duration_ms'.
    """
    passed = [r for r in results if r['status'] == 'pass']
    failed = [r for r in results if r['status'] == 'fail']
    return {
        'suite': suite_name,
        'total': len(results),
        'passed': len(passed),
        'failed': len(failed),
        'pass_rate': round(len(passed) / len(results) * 100, 1) if results else 0.0,
        'results': results,
    }

test_run = [
    {'name': 'test_boot',    'status': 'pass', 'duration_ms': 1200},
    {'name': 'test_network', 'status': 'fail', 'duration_ms': 3400},
    {'name': 'test_storage', 'status': 'pass', 'duration_ms': 890},
]

payload = format_test_results_payload('HW Validation Suite', test_run)
print(f"Suite: {payload['suite']}")
print(f"Pass rate: {payload['pass_rate']}%")  # 66.7%

### Practice: Validate API Response Schema

Write a function that validates an API response dict against a list of required keys. Return a list of missing keys. Return an empty list if all required keys are present.

**Function signature:** `def validate_schema(response: dict, required_keys: list[str]) -> list[str]:`

**Examples:**

- Input: `{'id': 1, 'name': 'Alice'}, ['id', 'name', 'email']` → Output: `['email']`
- Input: `{'id': 1, 'name': 'Alice', 'email': 'a@b.com'}, ['id', 'name', 'email']` → Output: `[]`

**Hints:**

- Use a list comprehension: [k for k in required_keys if k not in response].

In [ ]:
# TODO: Implement your solution
def validate_schema(response: dict, required_keys: list[str]) -> list[str]:
    pass

In [ ]:
# Solution: Validate API Response Schema
def validate_schema(response: dict, required_keys: list[str]) -> list[str]:
    """Return list of required keys missing from response."""
    return [k for k in required_keys if k not in response]

In [ ]:
# Tests for Validate API Response Schema
assert validate_schema({'id': 1, 'name': 'Alice'}, ['id', 'name', 'email']) == ['email']
assert validate_schema({'id': 1, 'name': 'Alice', 'email': 'a@b.com'}, ['id', 'name', 'email']) == []
assert validate_schema({}, ['id', 'name']) == ['id', 'name']
assert validate_schema({'x': 1}, []) == []
print('All tests passed!')

### Practice: Build Test Result Report

Given a list of test result dicts (each with 'name', 'status' ('pass'/'fail'/'skip'), and 'duration_ms'), return a summary dict with 'total', 'passed', 'failed', 'skipped', and 'pass_rate' (float, percentage rounded to 1 decimal). pass_rate = passed / (passed + failed) * 100, or 0.0 if no pass/fail.

**Function signature:** `def build_report(results: list[dict]) -> dict:`

**Examples:**

- Input: `[{'name': 'A', 'status': 'pass', 'duration_ms': 100}, {'name': 'B', 'status': 'fail', 'duration_ms': 200}, {'name': 'C', 'status': 'skip', 'duration_ms': 0}]` → Output: `{'total': 3, 'passed': 1, 'failed': 1, 'skipped': 1, 'pass_rate': 50.0}`

**Hints:**

- Use sum() with generator expressions to count each status.
- Guard against division by zero when computing pass_rate.

In [ ]:
# TODO: Implement your solution
def build_report(results: list[dict]) -> dict:
    pass

In [ ]:
# Solution: Build Test Result Report
def build_report(results: list[dict]) -> dict:
    """Build a test result summary report."""
    passed  = sum(1 for r in results if r['status'] == 'pass')
    failed  = sum(1 for r in results if r['status'] == 'fail')
    skipped = sum(1 for r in results if r['status'] == 'skip')
    denominator = passed + failed
    pass_rate = round(passed / denominator * 100, 1) if denominator else 0.0
    return {
        'total':     len(results),
        'passed':    passed,
        'failed':    failed,
        'skipped':   skipped,
        'pass_rate': pass_rate,
    }

In [ ]:
# Tests for Build Test Result Report
results = [
    {'name': 'A', 'status': 'pass', 'duration_ms': 100},
    {'name': 'B', 'status': 'fail', 'duration_ms': 200},
    {'name': 'C', 'status': 'skip', 'duration_ms': 0},
]
report = build_report(results)
assert report['total'] == 3
assert report['passed'] == 1
assert report['failed'] == 1
assert report['skipped'] == 1
assert report['pass_rate'] == 50.0
assert build_report([])['pass_rate'] == 0.0
print('All tests passed!')

### Practice: Redfish API — Query Server Health Across a Fleet

**NVIDIA SDET context:** Redfish is the REST API for server hardware management (BMC/iDRAC/iLO). It follows standard HTTP conventions and returns JSON.

Given a list of Redfish System resource dicts (simulating GET /redfish/v1/Systems/{id} responses), write a function `fleet_health_summary(systems)` that returns a dict with:
- `'total'`: total system count
- `'healthy'`: count where Status.Health == 'OK'
- `'degraded'`: count where Status.Health == 'Warning'
- `'critical'`: count where Status.Health == 'Critical'
- `'powered_on'`: count where PowerState == 'On'
- `'unhealthy_ids'`: list of system IDs where Health != 'OK'

**Function signature:** `def fleet_health_summary(systems: list[dict]) -> dict:`

**Examples:**

- Input: `[{'Id': 'sys1', 'PowerState': 'On', 'Status': {'Health': 'OK'}}, {'Id': 'sys2', 'PowerState': 'On', 'Status': {'Health': 'Warning'}}, {'Id': 'sys3', 'PowerState': 'Off', 'Status': {'Health': 'Critical'}}]` → Output: `{'total': 3, 'healthy': 1, 'degraded': 1, 'critical': 1, 'powered_on': 2, 'unhealthy_ids': ['sys2', 'sys3']}`

**Hints:**

- Use .get('Status', {}).get('Health', 'Unknown') for safe nested access.
- Append to unhealthy_ids for both Warning and Critical health states.

In [ ]:
# TODO: Implement your solution
def fleet_health_summary(systems: list[dict]) -> dict:
    pass

In [ ]:
# Solution: Redfish API — Query Server Health Across a Fleet
def fleet_health_summary(systems: list[dict]) -> dict:
    """Summarise Redfish system health across a server fleet."""
    summary = {'total': len(systems), 'healthy': 0, 'degraded': 0,
               'critical': 0, 'powered_on': 0, 'unhealthy_ids': []}
    for s in systems:
        health = s.get('Status', {}).get('Health', 'Unknown')
        if health == 'OK':
            summary['healthy'] += 1
        elif health == 'Warning':
            summary['degraded'] += 1
            summary['unhealthy_ids'].append(s.get('Id', 'unknown'))
        else:
            summary['critical'] += 1
            summary['unhealthy_ids'].append(s.get('Id', 'unknown'))
        if s.get('PowerState') == 'On':
            summary['powered_on'] += 1
    return summary

In [ ]:
# Tests for Redfish API — Query Server Health Across a Fleet
systems = [
    {'Id': 'sys1', 'PowerState': 'On',  'Status': {'Health': 'OK'}},
    {'Id': 'sys2', 'PowerState': 'On',  'Status': {'Health': 'Warning'}},
    {'Id': 'sys3', 'PowerState': 'Off', 'Status': {'Health': 'Critical'}},
    {'Id': 'sys4', 'PowerState': 'On',  'Status': {'Health': 'OK'}},
]
result = fleet_health_summary(systems)
assert result['total'] == 4
assert result['healthy'] == 2
assert result['degraded'] == 1
assert result['critical'] == 1
assert result['powered_on'] == 3
assert set(result['unhealthy_ids']) == {'sys2', 'sys3'}
assert fleet_health_summary([])['total'] == 0
print('All tests passed!')

### Key Takeaways

- API test scripts combine HTTP calls with assertions to validate behavior.
- Use mock data or unittest.mock.patch for offline/unit testing.
- CI/CD webhook handlers parse build event payloads and extract key fields.
- Test result reporting clients format and POST results to dashboards.
- HackerRank REST problems: fetch all pages, filter/aggregate, return result.

## Timed Practice — Mock Test

**Time limit: 25 minutes**

**Instructions:**

1. Set a timer before you begin.
2. Attempt **all** questions before checking any solutions.
3. Mark questions you are unsure about and revisit them if time permits.
4. When the timer ends, stop and review your answers against the solutions below.

### Practice: Fetch and Filter Paginated Users

**Timed Problem — target: ~8 minutes**

You are given a function `get_page(page: int) -> list[dict]` that returns a page of user dicts. Each user has 'id' (int), 'name' (str), 'active' (bool), and 'score' (float). The function returns an empty list when there are no more pages.

Write a function `top_active_users(get_page, n: int) -> list[str]` that fetches ALL pages, filters to active users only, and returns the names of the top `n` users by score (highest first).

**Function signature:** `def top_active_users(get_page, n: int) -> list[str]:`

**Examples:**

- Input: `get_page returning pages of users, n=2` → Output: `['Alice', 'Carol']  # top 2 active users by score`

**Hints:**

- Step 1: Fetch all pages with a while loop (break on empty list).
- Step 2: Filter to active=True users.
- Step 3: Sort by score descending, slice to n, extract names.

In [ ]:
# TODO: Implement your solution
def top_active_users(get_page, n: int) -> list[str]:
    pass

In [ ]:
# Solution: Fetch and Filter Paginated Users
def top_active_users(get_page, n: int) -> list[str]:
    """Fetch all pages, filter active users, return top n by score."""
    all_users = []
    page = 1
    while True:
        users = get_page(page)
        if not users:
            break
        all_users.extend(users)
        page += 1
    active = [u for u in all_users if u.get('active')]
    active.sort(key=lambda u: u['score'], reverse=True)
    return [u['name'] for u in active[:n]]

In [ ]:
# Tests for Fetch and Filter Paginated Users
pages_data = [
    [
        {'id': 1, 'name': 'Alice', 'active': True,  'score': 95.0},
        {'id': 2, 'name': 'Bob',   'active': False, 'score': 88.0},
    ],
    [
        {'id': 3, 'name': 'Carol', 'active': True,  'score': 91.0},
        {'id': 4, 'name': 'Dave',  'active': True,  'score': 72.0},
    ],
    [],  # end of pages
]
def get_page(p): return pages_data[p - 1] if p <= len(pages_data) else []

assert top_active_users(get_page, 2) == ['Alice', 'Carol']
assert top_active_users(get_page, 1) == ['Alice']
assert top_active_users(get_page, 10) == ['Alice', 'Carol', 'Dave']
print('All tests passed!')

### Practice: Aggregate API Metrics Across Pages

**Timed Problem — target: ~9 minutes**

You are given a function `get_metrics_page(page: int) -> list[dict]` that returns pages of API call records. Each record has 'endpoint' (str), 'status_code' (int), and 'response_time_ms' (float). Returns empty list when done.

Write `api_summary(get_metrics_page) -> dict` that fetches all pages and returns:
- `'total_calls'`: total number of records
- `'error_rate'`: percentage of calls with status >= 400 (rounded to 1 decimal)
- `'avg_response_ms'`: average response time in ms (rounded to 1 decimal)
- `'slowest_endpoint'`: endpoint with highest average response time

Return `{'total_calls': 0, 'error_rate': 0.0, 'avg_response_ms': 0.0, 'slowest_endpoint': None}` for empty data.

**Function signature:** `def api_summary(get_metrics_page) -> dict:`

**Examples:**

- Input: `get_metrics_page returning 2 pages of records` → Output: `{'total_calls': 4, 'error_rate': 25.0, 'avg_response_ms': 175.0, 'slowest_endpoint': '/api/data'}`

**Hints:**

- Step 1: Collect all records with a pagination loop.
- Step 2: Count errors (status >= 400) and compute error_rate.
- Step 3: Compute avg_response_ms across all records.
- Step 4: Group response times by endpoint, compute averages, find max.

In [ ]:
# TODO: Implement your solution
def api_summary(get_metrics_page) -> dict:
    pass

In [ ]:
# Solution: Aggregate API Metrics Across Pages
def api_summary(get_metrics_page) -> dict:
    """Aggregate API metrics across all pages."""
    all_records = []
    page = 1
    while True:
        records = get_metrics_page(page)
        if not records:
            break
        all_records.extend(records)
        page += 1

    if not all_records:
        return {'total_calls': 0, 'error_rate': 0.0,
                'avg_response_ms': 0.0, 'slowest_endpoint': None}

    total = len(all_records)
    errors = sum(1 for r in all_records if r['status_code'] >= 400)
    error_rate = round(errors / total * 100, 1)
    avg_rt = round(sum(r['response_time_ms'] for r in all_records) / total, 1)

    # Slowest endpoint by average response time
    ep_times: dict[str, list[float]] = {}
    for r in all_records:
        ep_times.setdefault(r['endpoint'], []).append(r['response_time_ms'])
    ep_avgs = {ep: sum(ts) / len(ts) for ep, ts in ep_times.items()}
    slowest = max(ep_avgs, key=ep_avgs.get)

    return {
        'total_calls': total,
        'error_rate': error_rate,
        'avg_response_ms': avg_rt,
        'slowest_endpoint': slowest,
    }

In [ ]:
# Tests for Aggregate API Metrics Across Pages
pages = [
    [
        {'endpoint': '/api/users', 'status_code': 200, 'response_time_ms': 100.0},
        {'endpoint': '/api/data',  'status_code': 200, 'response_time_ms': 300.0},
    ],
    [
        {'endpoint': '/api/users', 'status_code': 404, 'response_time_ms': 50.0},
        {'endpoint': '/api/data',  'status_code': 200, 'response_time_ms': 250.0},
    ],
    [],
]
def get_page(p): return pages[p - 1] if p <= len(pages) else []

result = api_summary(get_page)
assert result['total_calls'] == 4
assert result['error_rate'] == 25.0
assert result['avg_response_ms'] == 175.0
assert result['slowest_endpoint'] == '/api/data'
assert api_summary(lambda p: [])['total_calls'] == 0
print('All tests passed!')

### Practice: Join Users and Posts from Paginated APIs

**Timed Problem — target: ~8 minutes**

You have two data sources (simulated as lists):
- `users`: list of dicts with 'id' and 'name'
- `posts`: list of dicts with 'id', 'userId', and 'title'

Write `user_post_counts(users, posts) -> list[dict]` that returns a list of dicts with 'name' and 'post_count' for each user, sorted by post_count descending. Users with 0 posts should be included.

**Function signature:** `def user_post_counts(users: list[dict], posts: list[dict]) -> list[dict]:`

**Examples:**

- Input: `users=[{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}], posts=[{'id': 1, 'userId': 1, 'title': 'A'}, {'id': 2, 'userId': 1, 'title': 'B'}, {'id': 3, 'userId': 2, 'title': 'C'}]` → Output: `[{'name': 'Alice', 'post_count': 2}, {'name': 'Bob', 'post_count': 1}]`

**Hints:**

- Build a userId -> count dict from the posts list.
- Use counts.get(user_id, 0) to handle users with no posts.
- Sort the result list by post_count descending.

In [ ]:
# TODO: Implement your solution
def user_post_counts(users: list[dict], posts: list[dict]) -> list[dict]:
    pass

In [ ]:
# Solution: Join Users and Posts from Paginated APIs
def user_post_counts(users: list[dict], posts: list[dict]) -> list[dict]:
    """Return users with their post counts, sorted by count descending."""
    # Build a count map: userId -> count
    counts: dict[int, int] = {}
    for post in posts:
        uid = post['userId']
        counts[uid] = counts.get(uid, 0) + 1

    result = [
        {'name': u['name'], 'post_count': counts.get(u['id'], 0)}
        for u in users
    ]
    result.sort(key=lambda x: x['post_count'], reverse=True)
    return result

In [ ]:
# Tests for Join Users and Posts from Paginated APIs
users = [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}, {'id': 3, 'name': 'Carol'}]
posts = [
    {'id': 1, 'userId': 1, 'title': 'A'},
    {'id': 2, 'userId': 1, 'title': 'B'},
    {'id': 3, 'userId': 2, 'title': 'C'},
]
result = user_post_counts(users, posts)
assert result[0] == {'name': 'Alice', 'post_count': 2}
assert result[1] == {'name': 'Bob',   'post_count': 1}
assert result[2] == {'name': 'Carol', 'post_count': 0}
assert user_post_counts([], []) == []
print('All tests passed!')

## Quick-Reference Cheat Sheet

## REST API Coding — Quick Reference Cheat Sheet

---

### HTTP Status Codes

| Code | Name | Meaning |
|------|------|---------|
| 200  | OK | Request succeeded |
| 201  | Created | Resource created (POST) |
| 204  | No Content | Success, no body (DELETE) |
| 301  | Moved Permanently | Redirect |
| 304  | Not Modified | Cached response still valid |
| 400  | Bad Request | Invalid request syntax/params |
| 401  | Unauthorized | Not authenticated |
| 403  | Forbidden | Authenticated but not permitted |
| 404  | Not Found | Resource doesn't exist |
| 429  | Too Many Requests | Rate limited — back off |
| 500  | Internal Server Error | Server-side bug |
| 502  | Bad Gateway | Upstream server error |
| 503  | Service Unavailable | Server overloaded/down |
| 504  | Gateway Timeout | Upstream timeout |

---

### Common `requests` Patterns

```python
import requests

# GET with query params
r = requests.get(url, params={'page': 1, 'limit': 20}, timeout=10)

# POST with JSON body
r = requests.post(url, json={'key': 'value'}, timeout=10)

# PUT (replace resource)
r = requests.put(url, json={'key': 'new_value'}, timeout=10)

# DELETE
r = requests.delete(url, timeout=10)

# With auth headers
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
r = requests.get(url, headers=headers, timeout=10)

# Check response
if r.status_code == 200:
    data = r.json()   # dict or list
r.raise_for_status()  # raises HTTPError for 4xx/5xx
```

---

### JSON Handling

```python
import json

# Parse JSON string
data = json.loads(json_string)

# Serialize to JSON string
s = json.dumps(data, indent=2)

# Safe nested access
val = data.get('key', {}).get('nested', 'default')

# Deep get helper
def deep_get(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict): return default
        d = d.get(k, default)
        if d is default: return default
    return d
```

---

### Pagination Patterns

```python
# Page-based (most common in HackerRank)
all_items, page = [], 1
while True:
    items = get_page(page)   # returns [] on last page
    if not items: break
    all_items.extend(items)
    page += 1

# Offset-based
all_items, offset, limit = [], 0, 20
while True:
    items = get_items(offset=offset, limit=limit)
    if not items: break
    all_items.extend(items)
    offset += limit

# Cursor-based
all_items, cursor = [], None
while True:
    data = get_data(cursor=cursor)
    all_items.extend(data['items'])
    cursor = data.get('next_cursor')
    if not cursor: break
```

---

### Common HackerRank REST API Patterns

**Pattern 1: Filter paginated results**
```python
# Fetch all pages, filter by condition, return matching items
def get_active_users(get_page):
    all_users, page = [], 1
    while True:
        users = get_page(page)
        if not users: break
        all_users.extend(users)
        page += 1
    return [u for u in all_users if u.get('active')]
```

**Pattern 2: Aggregate across all pages**
```python
# Fetch all pages, compute sum/average/max
def total_score(get_page):
    total, page = 0, 1
    while True:
        items = get_page(page)
        if not items: break
        total += sum(item['score'] for item in items)
        page += 1
    return total
```

**Pattern 3: Join two endpoints**
```python
# Build lookup from one endpoint, enrich with another
user_map = {u['id']: u['name'] for u in fetch_all_users()}
posts = fetch_all_posts()
enriched = [{**p, 'author': user_map.get(p['userId'], 'Unknown')} for p in posts]
```

**Pattern 4: Top-N from paginated data**
```python
# Fetch all, sort, slice
all_items = fetch_all_pages(get_page)
top_n = sorted(all_items, key=lambda x: x['score'], reverse=True)[:n]
```

---

### Authentication Quick Reference

```python
# Bearer token
headers = {'Authorization': f'Bearer {token}'}

# API key in header
headers = {'X-API-Key': api_key}

# Basic auth
import requests
r = requests.get(url, auth=('username', 'password'))

# Redfish (Basic auth)
r = requests.get(
    'https://bmc-host/redfish/v1/Systems',
    auth=('admin', 'password'),
    verify=False,  # self-signed cert in lab
    timeout=10
)
```

---

### Error Handling Template

```python
def safe_get(url, headers=None, params=None, max_retries=3):
    retryable = {429, 500, 502, 503, 504}
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=10)
            if r.status_code == 200:
                return r.json()
            if r.status_code not in retryable:
                raise ValueError(f'HTTP {r.status_code}')
        except requests.Timeout:
            pass  # retry on timeout
    raise RuntimeError(f'Failed after {max_retries} attempts')
```